# Introduction à **PySpark**

**Dataset:** [ventes.csv](https://drive.google.com/file/d/1kzNj66aWbtnOYZIwTj2nux-knLp2bETs/view?usp=drive_link) \
Cet ensemble de données contient des informations fictives sur des ventes réalisées par une entreprise, chaque ligne correspond à une vente.

*Voir aussi le [Notions](https://www.notion.so/Spark-Pyspark-27d7bf1b1fa180419888f91f351aec50?source=copy_link) pour en savoir plus sur **Spark** et **PySpark**.*

## Nettoyer et transformer des données avec **PySpark**

### Initialisation d'une session **PySpark**

Une session est le point d'entrée pour manipuler des données avec **PySpark**.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/10/02 10:38:06 WARN Utils: Your hostname, gattano-ThinkPad-P53, resolves to a loopback address: 127.0.1.1; using 192.168.1.130 instead (on interface enx00e04c680450)
25/10/02 10:38:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/02 10:38:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Chargement des données

En l'absence d'information sur le type dans l'en-tête du fichier `csv` le lecteur `csv` de **PySpark** semble considérer le type `string` par défaut. L'argument `schema` permet de lui donner l'information.

In [ ]:
ventes_schema = \
    'id_transaction INT, ' + \
    'client_nom STRING, ' + \
    'client_age INT, ' + \
    'client_ville STRING, ' + \
    'produit_nom STRING, ' + \
    'produit_categorie STRING, ' + \
    'produit_marque STRING, ' + \
    'prix_catalogue INT, ' + \
    'magasin_nom STRING, ' + \
    'magasin_type STRING, ' + \
    'magasin_region STRING,' + \
    'date DATE, ' + \
    'quantite INT, ' + \
    'montant_total INT'

spark.read.csv('../data/ventes.csv', schema=ventes_schema, header=True).show()

+--------------+----------+----------+------------+----------------+-----------------+--------------+--------------+--------------+------------+--------------------+----------+--------+-------------+
|id_transaction|client_nom|client_age|client_ville|     produit_nom|produit_categorie|produit_marque|prix_catalogue|   magasin_nom|magasin_type|      magasin_region|      date|quantite|montant_total|
+--------------+----------+----------+------------+----------------+-----------------+--------------+--------------+--------------+------------+--------------------+----------+--------+-------------+
|             1|     Alice|        25|       Paris|      Ordinateur|     Informatique|          Dell|           800| Boutique Lyon|    Physique|Auvergne-Rhône-Alpes|2023-03-12|       2|         NULL|
|             2|       Bob|        34|        Lyon|      Smartphone|       Téléphonie|         Apple|          1200| Boutique Lyon|    Physique|Auvergne-Rhône-Alpes|2023-01-27|       5|         NULL|


Récupération des données :

In [3]:
ventes = spark.read.csv('data/ventes.csv', schema=ventes_schema, header=True)

### Visualisation générale des données

Réglage pour un affichage automatique (pas besoin de préciser le .show par la suite)

In [4]:
spark.conf.set('spark.sql.repl.eagerEval.enabled', True)
spark.conf.set('spark.sql.repl.eagerEval.maxNumRows', 10)
ventes

id_transaction,client_nom,client_age,client_ville,produit_nom,produit_categorie,produit_marque,prix_catalogue,magasin_nom,magasin_type,magasin_region,date,quantite,montant_total
1,Alice,25,Paris,Ordinateur,Informatique,Dell,800,Boutique Lyon,Physique,Auvergne-Rhône-Alpes,2023-03-12,2,NULL
2,Bob,34,Lyon,Smartphone,Téléphonie,Apple,1200,Boutique Lyon,Physique,Auvergne-Rhône-Alpes,2023-01-27,5,NULL
3,Alice,25,Paris,Montre connectée,Accessoires,Garmin,300,E-Shop,En ligne,National,2023-01-09,1,NULL
4,Alice,25,Paris,Smartphone,Téléphonie,Apple,1200,Boutique Paris,Physique,Île-de-France,2023-05-10,5,NULL
5,Alice,25,Paris,Montre connectée,Accessoires,Garmin,300,Boutique Paris,Physique,Île-de-France,2023-06-16,5,NULL
6,David,40,Bordeaux,Smartphone,Téléphonie,Apple,1200,E-Shop,En ligne,National,2023-05-31,3,NULL
7,Alice,25,Paris,Smartphone,Téléphonie,Apple,1200,Boutique Lyon,Physique,Auvergne-Rhône-Alpes,2023-04-19,3,NULL
8,Charlie,29,Marseille,Smartphone,Téléphonie,Apple,1200,Boutique Paris,Physique,Île-de-France,2023-03-28,1,NULL
9,Alice,25,Paris,Tablette,Informatique,Samsung,600,Boutique Paris,Physique,Île-de-France,2023-04-02,3,NULL
10,Emma,31,Toulouse,Casque audio,Accessoires,Sony,150,Boutique Paris,Physique,Île-de-France,2023-04-28,5,NULL


Alternative d'affichage

In [5]:
ventes.show(1,vertical=True)

-RECORD 0---------------------------------
 id_transaction    | 1                    
 client_nom        | Alice                
 client_age        | 25                   
 client_ville      | Paris                
 produit_nom       | Ordinateur           
 produit_categorie | Informatique         
 produit_marque    | Dell                 
 prix_catalogue    | 800                  
 magasin_nom       | Boutique Lyon        
 magasin_type      | Physique             
 magasin_region    | Auvergne-Rhône-Alpes 
 date              | 2023-03-12           
 quantite          | 2                    
 montant_total     | NULL                 
only showing top 1 row


Visualisation de la structure de la table

In [6]:
ventes.columns

['id_transaction',
 'client_nom',
 'client_age',
 'client_ville',
 'produit_nom',
 'produit_categorie',
 'produit_marque',
 'prix_catalogue',
 'magasin_nom',
 'magasin_type',
 'magasin_region',
 'date',
 'quantite',
 'montant_total']

In [7]:
ventes.printSchema()

root
 |-- id_transaction: integer (nullable = true)
 |-- client_nom: string (nullable = true)
 |-- client_age: integer (nullable = true)
 |-- client_ville: string (nullable = true)
 |-- produit_nom: string (nullable = true)
 |-- produit_categorie: string (nullable = true)
 |-- produit_marque: string (nullable = true)
 |-- prix_catalogue: integer (nullable = true)
 |-- magasin_nom: string (nullable = true)
 |-- magasin_type: string (nullable = true)
 |-- magasin_region: string (nullable = true)
 |-- date: date (nullable = true)
 |-- quantite: integer (nullable = true)
 |-- montant_total: integer (nullable = true)



Obtenir quelques statistiques sur les colonnes

In [8]:
ventes.describe().show()

25/10/02 10:39:17 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+-----------------+------------------+------------+------------+-----------------+--------------+-----------------+------------------+------------+--------------------+------------------+-------------+
|summary|    id_transaction|       client_nom|        client_age|client_ville| produit_nom|produit_categorie|produit_marque|   prix_catalogue|       magasin_nom|magasin_type|      magasin_region|          quantite|montant_total|
+-------+------------------+-----------------+------------------+------------+------------+-----------------+--------------+-----------------+------------------+------------+--------------------+------------------+-------------+
|  count|               495|              495|               495|         495|         495|              495|           495|              495|               495|         495|                 495|               487|            0|
|   mean|             248.0|             45.0|130.16363636363636|        NULL|      

### Sélection et visualisation des données

Sélection d'une colonne

In [9]:
ventes.client_nom

Column<'client_nom'>

In [10]:
ventes.select(ventes.client_nom)

client_nom
Alice
Bob
Alice
Alice
Alice
David
Alice
Charlie
Alice
Emma


Accès conditionnel des données

In [11]:
ventes.filter(ventes.client_nom == 'Bob')

id_transaction,client_nom,client_age,client_ville,produit_nom,produit_categorie,produit_marque,prix_catalogue,magasin_nom,magasin_type,magasin_region,date,quantite,montant_total
2,Bob,34,Lyon,Smartphone,Téléphonie,Apple,1200,Boutique Lyon,Physique,Auvergne-Rhône-Alpes,2023-01-27,5,NULL
15,Bob,34,Lyon,Casque audio,Accessoires,Sony,150,E-Shop,En ligne,National,2023-02-23,3,NULL
17,Bob,34,Lyon,Smartphone,3,Apple,1200,E-Shop,En ligne,National,2023-04-08,3,NULL
19,Bob,34,Lyon,Ordinateur,Informatique,Dell,800,E-Shop,En ligne,National,2023-04-13,3,NULL
31,Bob,34,Lyon,Montre connectée,Accessoires,Garmin,300,Boutique Paris,Physique,Île-de-France,2023-06-10,3,NULL
33,Bob,34,Lyon,Montre connectée,Accessoires,Garmin,300,Boutique Lyon,Physique,Auvergne-Rhône-Alpes,2023-01-01,5,NULL
42,Bob,34,Lyon,Ordinateur,Informatique,Dell,800,E-Shop,En ligne,National,2023-01-06,5,NULL
48,Bob,34,Lyon,Ordinateur,Informatique,Dell,800,Boutique Paris,Physique,Île-de-France,2023-06-18,4,NULL
51,Bob,34,Lyon,Smartphone,Téléphonie,Apple,1200,Boutique Paris,Physique,Île-de-France,2023-05-18,4,NULL
52,Bob,34,Lyon,Tablette,Informatique,Samsung,600,Boutique Paris,Physique,Île-de-France,2023-03-13,4,NULL


### Manipulation des données

Déclarer une nouvelle colonne

In [12]:
from pyspark.sql.functions import upper

ventes.withColumn('client_maj_nom', upper(ventes.client_nom)).select("client_nom", "client_maj_nom")

client_nom,client_maj_nom
Alice,ALICE
Bob,BOB
Alice,ALICE
Alice,ALICE
Alice,ALICE
David,DAVID
Alice,ALICE
Charlie,CHARLIE
Alice,ALICE
Emma,EMMA


***Remarque:*** *La précédente commande n'a pas modifié **ventes***

In [13]:
ventes

id_transaction,client_nom,client_age,client_ville,produit_nom,produit_categorie,produit_marque,prix_catalogue,magasin_nom,magasin_type,magasin_region,date,quantite,montant_total
1,Alice,25,Paris,Ordinateur,Informatique,Dell,800,Boutique Lyon,Physique,Auvergne-Rhône-Alpes,2023-03-12,2,NULL
2,Bob,34,Lyon,Smartphone,Téléphonie,Apple,1200,Boutique Lyon,Physique,Auvergne-Rhône-Alpes,2023-01-27,5,NULL
3,Alice,25,Paris,Montre connectée,Accessoires,Garmin,300,E-Shop,En ligne,National,2023-01-09,1,NULL
4,Alice,25,Paris,Smartphone,Téléphonie,Apple,1200,Boutique Paris,Physique,Île-de-France,2023-05-10,5,NULL
5,Alice,25,Paris,Montre connectée,Accessoires,Garmin,300,Boutique Paris,Physique,Île-de-France,2023-06-16,5,NULL
6,David,40,Bordeaux,Smartphone,Téléphonie,Apple,1200,E-Shop,En ligne,National,2023-05-31,3,NULL
7,Alice,25,Paris,Smartphone,Téléphonie,Apple,1200,Boutique Lyon,Physique,Auvergne-Rhône-Alpes,2023-04-19,3,NULL
8,Charlie,29,Marseille,Smartphone,Téléphonie,Apple,1200,Boutique Paris,Physique,Île-de-France,2023-03-28,1,NULL
9,Alice,25,Paris,Tablette,Informatique,Samsung,600,Boutique Paris,Physique,Île-de-France,2023-04-02,3,NULL
10,Emma,31,Toulouse,Casque audio,Accessoires,Sony,150,Boutique Paris,Physique,Île-de-France,2023-04-28,5,NULL


Application de fonction via la représentation des données en pandas

In [14]:
import pandas as pd
from pyspark.sql.functions import pandas_udf

@pandas_udf('long')
def pandas_plus_one(series: pd.Series) -> pd.Series:
    # Simply plus one by using pandas Series.
    return series + 1

ventes.select(pandas_plus_one(ventes.id_transaction)).show()


+-------------------------------+
|pandas_plus_one(id_transaction)|
+-------------------------------+
|                              2|
|                              3|
|                              4|
|                              5|
|                              6|
|                              7|
|                              8|
|                              9|
|                             10|
|                             11|
|                             12|
|                             13|
|                             14|
|                             15|
|                             16|
|                             17|
|                             18|
|                             19|
|                             20|
|                             21|
+-------------------------------+
only showing top 20 rows


Application de fonction sur le dataframe complet. Exemple redéfinit un filtrage de données

In [15]:
def pandas_filter_func(iterator):
    for pandas_df in iterator:
        yield pandas_df[(pandas_df.client_nom == 'Bob') & (pandas_df.produit_nom == 'Smartphone')]

ventes.mapInPandas(pandas_filter_func, schema=ventes.schema).show()

+--------------+----------+----------+------------+-----------+-----------------+--------------+--------------+--------------+------------+--------------------+----------+--------+-------------+
|id_transaction|client_nom|client_age|client_ville|produit_nom|produit_categorie|produit_marque|prix_catalogue|   magasin_nom|magasin_type|      magasin_region|      date|quantite|montant_total|
+--------------+----------+----------+------------+-----------+-----------------+--------------+--------------+--------------+------------+--------------------+----------+--------+-------------+
|             2|       Bob|        34|        Lyon| Smartphone|       Téléphonie|         Apple|          1200| Boutique Lyon|    Physique|Auvergne-Rhône-Alpes|2023-01-27|       5|         NULL|
|            17|       Bob|        34|        Lyon| Smartphone|                3|         Apple|          1200|        E-Shop|    En ligne|            National|2023-04-08|       3|         NULL|
|            51|       Bo

Agrégation de données

In [16]:
ventes.select('client_nom', 'prix_catalogue').groupby('client_nom').sum()

client_nom,sum(prix_catalogue)
78,800
Charlie,62750
Bob,53450
Alice,55700
Emma,64000
12,600
David,57850


In [17]:
def get_montant_total(pandas_df):
    return pandas_df.assign(montant_total=pandas_df.prix_catalogue * pandas_df.quantite)

ventes.groupby('client_nom').applyInPandas(get_montant_total, schema=ventes.schema)


id_transaction,client_nom,client_age,client_ville,produit_nom,produit_categorie,produit_marque,prix_catalogue,magasin_nom,magasin_type,magasin_region,date,quantite,montant_total
210,12,40,Bordeaux,Tablette,Informatique,Samsung,600,789,Physique,Île-de-France,2023-04-27,5,3000
146,78,31,Toulouse,Ordinateur,Informatique,Dell,800,E-Shop,En ligne,National,2023-05-17,4,3200
1,Alice,25,Paris,Ordinateur,Informatique,Dell,800,Boutique Lyon,Physique,Auvergne-Rhône-Alpes,2023-03-12,2,1600
3,Alice,25,Paris,Montre connectée,Accessoires,Garmin,300,E-Shop,En ligne,National,2023-01-09,1,300
4,Alice,25,Paris,Smartphone,Téléphonie,Apple,1200,Boutique Paris,Physique,Île-de-France,2023-05-10,5,6000
5,Alice,25,Paris,Montre connectée,Accessoires,Garmin,300,Boutique Paris,Physique,Île-de-France,2023-06-16,5,1500
7,Alice,25,Paris,Smartphone,Téléphonie,Apple,1200,Boutique Lyon,Physique,Auvergne-Rhône-Alpes,2023-04-19,3,3600
9,Alice,25,Paris,Tablette,Informatique,Samsung,600,Boutique Paris,Physique,Île-de-France,2023-04-02,3,1800
11,Alice,25,Paris,Tablette,Informatique,Samsung,600,Boutique Paris,Physique,Île-de-France,2023-05-22,3,1800
13,Alice,25,Paris,Smartphone,Téléphonie,Apple,1200,E-Shop,En ligne,National,2023-01-21,2,2400


### Requête SQL

Pour permettre l'utilisation de requêtes SQL, il faut d'abord mettre le dataframe dans une base temporaire.

In [18]:
ventes.createOrReplaceTempView("ventes")
spark.sql('SELECT count(*) FROM ventes')

count(1)
495


Les requêtes peuvent également invoquer les UDFs que nous avons définis

In [19]:
@pandas_udf("integer")
def add_one(s: pd.Series) -> pd.Series:
    return s + 1

spark.udf.register("add_one", add_one)
spark.sql("SELECT add_one(prix_catalogue) FROM ventes").show()

+-----------------------+
|add_one(prix_catalogue)|
+-----------------------+
|                    801|
|                   1201|
|                    301|
|                   1201|
|                    301|
|                   1201|
|                   1201|
|                   1201|
|                    601|
|                    151|
|                    601|
|                    151|
|                   1201|
|                    601|
|                    151|
|                    301|
|                   1201|
|                   1201|
|                    801|
|                   -849|
+-----------------------+
only showing top 20 rows
